## Outlier Target Selection (MIA Paper §4.3)

Selection criterion from the paper:

> *"records that either have rare categorical attribute values or numerical values outside the attribute's 95% quantile"*

**Selection pipeline:**

1. **Numerical columns** — Flag records above the 95th percentile; score by relative excess
2. **Categorical columns** — Flag records with rare values (frequency < 1%); score by inverse frequency
3. **Score matrix** — Combine all column scores into a single matrix
4. **Diversity selection** — Define thematic column groups; pick the top-scoring record per group

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ── Parameters ─────────────────────────────────
DATA_DIR = "../data/adult"
QUANTILE        = 0.95   # percentile threshold for numerical outliers
RARE_THRESHOLD  = 0.01   # frequency threshold for rare categorical values
EXCLUDE_COLS    = ["label", "fnlwgt"]   # columns excluded from scoring
N_TARGETS       = 5      # number of outlier records to select

# ── Data loading ───────────────────────────────
# Assign sequential IDs compatible with SDR's load_local_data_as_df
df_raw = pd.read_csv(f"{DATA_DIR}.csv")
df_raw["ID"] = [f"ID{i}" for i in range(len(df_raw))]
df = df_raw.set_index("ID")

# ── Column type classification ──────────────────
feat_cols = [c for c in df.columns if c not in EXCLUDE_COLS]
num_cols  = [c for c in feat_cols if pd.api.types.is_numeric_dtype(df[c])]
cat_cols  = [c for c in feat_cols if c not in num_cols]

print(f"Feature columns: {len(feat_cols)}")
print(f"  Numerical  : {num_cols}")
print(f"  Categorical: {cat_cols}")

### Step 1 — Numerical columns: scoring by 95th-percentile excess

- **Criterion**: value > p95
- **Score**: `(value − p95) / p95`  (relative excess; columns with p95 = 0 use `value / max`)

In [ ]:
num_scores = pd.DataFrame(0.0, index=df.index, columns=num_cols)
num_thresholds = {}

for col in num_cols:
    p95 = df[col].quantile(QUANTILE)
    num_thresholds[col] = p95
    if p95 > 0:
        num_scores[col] = ((df[col] - p95) / p95).clip(lower=0)
    else:
        max_val = df[col].max()
        if max_val > 0:
            num_scores[col] = (df[col] / max_val).clip(lower=0)

# p95 threshold and number of records above it per column
summary = pd.DataFrame({
    "p95": num_thresholds,
    "# records above p95": (num_scores > 0).sum(),
    "max score": num_scores.max().round(3),
    "top record ID": num_scores.idxmax(),
}).rename_axis("column")

display(summary)

# Top 10 records by total score across all numerical columns
top_num = num_scores[num_scores.any(axis=1)].copy()
top_num["total score"] = top_num.sum(axis=1)
display(top_num.sort_values("total score", ascending=False).head(10).round(3))

### Step 2 — Categorical columns: scoring by rarity

- **Criterion**: value frequency < `RARE_THRESHOLD` (default 1%)
- **Score**: `1 / frequency`  (rarer values score higher)

In [ ]:
cat_scores = pd.DataFrame(0.0, index=df.index, columns=cat_cols)
cat_rare_vals = {}

for col in cat_cols:
    freq = df[col].value_counts(normalize=True)
    rare_vals = freq[freq < RARE_THRESHOLD]
    cat_rare_vals[col] = rare_vals
    freq_series = df[col].map(freq.to_dict()).astype(float)
    cat_scores[col] = np.where(
        freq_series < RARE_THRESHOLD,
        1.0 / freq_series.clip(lower=1e-10),
        0.0,
    )

# Summary table of rare values
rare_summary_rows = []
for col, rare_vals in cat_rare_vals.items():
    for val, freq in rare_vals.items():
        rare_summary_rows.append({
            "column": col, "rare value": val,
            "frequency (%)": round(freq * 100, 3),
            "# records": round(freq * len(df)),
            "score": round(1.0 / freq, 1),
        })

rare_df = pd.DataFrame(rare_summary_rows).sort_values("frequency (%)").reset_index(drop=True)
display(rare_df)

### Step 3 — Combined score matrix

Merge numerical and categorical scores into one matrix. Records with at least one non-zero score are **outlier candidates**.

> **Note:** The *total score* shown below (sum across all columns) is used **only for display** — to surface the most notable candidates visually.
> The actual selection in Steps 4–5 uses the **per-group maximum score**, not the total, so that records extreme in *one* dimension are preferred over records that are slightly above average in many dimensions.

In [ ]:
scores_raw = pd.concat([num_scores, cat_scores], axis=1)[feat_cols]

# Normalize each column to [0, 1] by dividing by its maximum score
# → unifies numerical scores (~19) and categorical scores (~32561) onto the same scale
col_max = scores_raw.max()
scores_norm = scores_raw.div(col_max.where(col_max > 0, 1.0))

# Tiebreaker: add a tiny fraction of the raw score so that when two records
# share the same normalized score, the one with higher raw rarity wins
global_max = scores_raw.max().max()
scores = scores_norm + scores_raw * (1e-6 / global_max) if global_max > 0 else scores_norm

is_candidate = scores.gt(0).any(axis=1)
candidates = scores[is_candidate].copy()
candidates["total score"] = candidates[feat_cols].sum(axis=1)

print(f"Outlier candidates: {is_candidate.sum()} / {len(df)} records")

# Top 15 candidates (display with [0,1] normalized scores)
display(
    scores_norm[is_candidate]
    .assign(**{"total score": candidates["total score"]})
    .sort_values("total score", ascending=False)
    .head(15)
    .round(3)
    .style.background_gradient(cmap="YlOrRd", axis=0, subset=feat_cols)
)

### Step 4 — Column groups and per-group top candidates

Group columns by theme and inspect the top-scoring record per group.
This ensures **one record per outlier type** is selected (diversity).

In [ ]:
col_groups = {
    "financial":   ["capital-gain", "capital-loss"],
    "work":        ["hours-per-week", "workclass", "occupation"],
    "demographic": ["age", "race", "native-country"],
    "education":   ["education", "education-num"],
    "family":      ["marital-status", "relationship"],
}

# Inspect the top-scoring record per group
remaining = scores[is_candidate].copy()

group_preview_rows = []
for group, cols in col_groups.items():
    valid_cols = [c for c in cols if c in remaining.columns]
    group_score = remaining[valid_cols].max(axis=1)
    best_id = group_score.idxmax()
    best_score = group_score.max()
    best_col = remaining.loc[best_id, valid_cols].idxmax()
    best_val = df.loc[best_id, best_col]
    group_preview_rows.append({
        "group": group,
        "primary column": best_col,
        "candidate ID": best_id,
        "value": best_val,
        "score": round(best_score, 3),
    })

display(pd.DataFrame(group_preview_rows).set_index("group"))

### Step 5 — Final selection: one record per group

Process groups ordered by fewest outlier candidates (rarest first) and pick the top-scoring record from each.
Already-selected records are removed from the pool before the next group is processed.

In [ ]:
remaining = scores[is_candidate].copy()
selected_ids = []
result_rows  = []

# After normalization all groups have max score ≈ 1.0, so sort by
# number of outlier candidates in the group (fewer = rarer = higher priority)
def group_candidate_count(g):
    cols = [c for c in col_groups[g] if c in remaining.columns]
    return (remaining[cols] > 0).any(axis=1).sum()

sorted_groups = sorted(col_groups, key=group_candidate_count)

for group in sorted_groups:
    if len(selected_ids) >= N_TARGETS:
        break
    valid_cols  = [c for c in col_groups[group] if c in remaining.columns]
    group_score = remaining[valid_cols].max(axis=1)
    if group_score.max() == 0:
        continue

    best_id  = group_score.idxmax()
    rec      = scores.loc[best_id]
    outlier_cols = rec[rec > 0].index.tolist()
    primary_col  = rec.idxmax()

    selected_ids.append(best_id)
    remaining = remaining.drop(best_id)  # remove selected record from pool

    result_rows.append({
        "group":           group,
        "selected ID":     best_id,
        "primary column":  primary_col,
        "value":           df.loc[best_id, primary_col],
        "norm. score":     round(rec[primary_col], 3),
        "outlier columns": ", ".join(outlier_cols),
    })

result_df = pd.DataFrame(result_rows).set_index("group")
display(result_df)

print("\nSelected IDs:", str(selected_ids).replace("'", '"'))

In [ ]:
# Inspect the raw data of selected records
display(df.loc[selected_ids])

# Distribution of each column

In [ ]:
df = pd.read_csv(f"{DATA_DIR}.csv")

# Convert object columns to category
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category')

# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include='category').columns.tolist()

all_cols = numeric_cols + categorical_cols
n = len(all_cols)

fig, axes = plt.subplots(
    nrows=(n + 2) // 3,
    ncols=3,
    figsize=(15, 4 * ((n + 2) // 3))
)
axes = axes.flatten()

# Numeric columns → histogram
for ax, col in zip(axes, numeric_cols):
    df[col].hist(bins=10, ax=ax)
    ax.set_title(col)

# Categorical columns → bar chart
for ax, col in zip(axes[len(numeric_cols):], categorical_cols):
    df[col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)

# Hide unused subplots
for ax in axes[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()